<a href="https://colab.research.google.com/github/ser2025/Aurora_Website/blob/master/02_Titanic_My_Model_Model3_Controlled_Features_ipynb_adl%C4%B1_not_defterinin_kopyas%C4%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Titanic — My Model 3
## Controlled Feature Engineering

### Amaç

Bu çalışmada, önceki modelde kullanılan feature engineering
yaklaşımının daha kontrollü ve sade bir versiyonu test edilecektir.

### Önceki Sonuçlar

- Baseline Validation Accuracy: %77,09
- Model 2 Validation Accuracy: %82,12
- Model 2 Kaggle Score: 0.77751

### Model 3 Yaklaşımı

Kullanılacak özellikler:

- Pclass
- Sex
- SibSp
- Parch
- Title
- FamilySize

Değişiklikler:

- Nadir Title değerleri `Rare` altında birleştirilecektir.
- `IsAlone` kullanılmayacaktır.
- `FarePerPerson` kullanılmayacaktır.

### Deneyin Amacı

Daha sade ve kontrollü bir feature setinin validation
performansına etkisini ölçmek ve Model 2 ile karşılaştırmaktır.

In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

base_path = "/content/drive/MyDrive/Yapay Zeka Eğitimi 2026/KAGGLE_500_PUAN/01_Raw_Data"

train_data = pd.read_csv(
    os.path.join(base_path, "ttrain.csv")
)

test_data = pd.read_csv(
    os.path.join(base_path, "ttest.csv")
)

print("Train:", train_data.shape)
print("Test:", test_data.shape)

Train: (891, 12)
Test: (418, 11)


In [ ]:
title = train_data["Name"].str.extract(
    r",\s*([^.]*)\.",
    expand=False
)

common_titles = ["Mr", "Miss", "Mrs", "Master"]

title_grouped = title.where(
    title.isin(common_titles),
    "Rare"
)

print(title_grouped.value_counts())

Name
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64


### Title Gruplama

Model 2'de bazı Title kategorileri çok az sayıda gözlem içeriyordu.

Bu modelde:

- Mr
- Miss
- Mrs
- Master

yaygın gruplar olarak korunmuş, diğer tüm unvanlar `Rare`
kategorisinde birleştirilmiştir.

Amaç, düşük frekanslı kategorilerin oluşturabileceği
aşırı uyum (overfitting) riskini azaltmayı test etmektir.

In [ ]:
family_size = train_data["SibSp"] + train_data["Parch"] + 1

In [ ]:
model3_data = pd.DataFrame({
    "Pclass": train_data["Pclass"],
    "Sex": train_data["Sex"],
    "SibSp": train_data["SibSp"],
    "Parch": train_data["Parch"],
    "Title": title_grouped,
    "FamilySize": family_size
})

X_model3 = pd.get_dummies(
    model3_data,
    drop_first=False
)

y = train_data["Survived"]

print("Model 3 X shape:", X_model3.shape)
display(X_model3.head())

Model 3 X shape: (891, 11)


,Pclass,SibSp,Parch,FamilySize,Sex_female,Sex_male,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,3,1,0,2,False,True,False,False,True,False,False
1,1,1,0,2,True,False,False,False,False,True,False
2,3,0,0,1,True,False,False,True,False,False,False
3,1,1,0,2,True,False,False,False,False,True,False
4,3,0,0,1,False,True,False,False,True,False,False


In [ ]:
X_train3, X_val3, y_train3, y_val3 = train_test_split(
    X_model3,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train3:", X_train3.shape)
print("X_val3:", X_val3.shape)

X_train3: (712, 11)
X_val3: (179, 11)


In [ ]:
model3 = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=1
)

model3.fit(
    X_train3,
    y_train3
)

RandomForestClassifier(max_depth=5, random_state=1)

In [ ]:
model3_pred = model3.predict(X_val3)

model3_accuracy = accuracy_score(
    y_val3,
    model3_pred
)

print(f"Model 3 Validation Accuracy: {model3_accuracy:.4f}")
print(f"Model 3 Validation Accuracy: %{model3_accuracy * 100:.2f}")

Model 3 Validation Accuracy: 0.8268
Model 3 Validation Accuracy: %82.68


## Model 3 — Validation Sonucu

Model 3, kontrollü feature engineering yaklaşımı ile
**%82,68 validation accuracy** elde etmiştir.

Model 2'nin validation accuracy değeri %82,12 olduğundan,
Model 3 **0,56 yüzde puanlık** bir iyileşme sağlamıştır.

Baseline model ile karşılaştırıldığında toplam iyileşme
**5,59 yüzde puanıdır**.

Model 3'te düşük frekanslı Title kategorileri `Rare` altında
birleştirilmiş, `FamilySize` korunmuş, `IsAlone` ve
`FarePerPerson` modelden çıkarılmıştır.

Bu sonuç, daha fazla özellik kullanmanın her zaman daha iyi
performans anlamına gelmediğini göstermektedir.

In [ ]:
final_model3 = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=1
)

final_model3.fit(
    X_model3,
    y
)

RandomForestClassifier(max_depth=5, random_state=1)

In [ ]:
test_title = test_data["Name"].str.extract(
    r",\s*([^.]*)\.",
    expand=False
)

common_titles = ["Mr", "Miss", "Mrs", "Master"]

test_title_grouped = test_title.where(
    test_title.isin(common_titles),
    "Rare"
)

test_family_size = test_data["SibSp"] + test_data["Parch"] + 1

In [ ]:
test_model3_data = pd.DataFrame({
    "Pclass": test_data["Pclass"],
    "Sex": test_data["Sex"],
    "SibSp": test_data["SibSp"],
    "Parch": test_data["Parch"],
    "Title": test_title_grouped,
    "FamilySize": test_family_size
})

X_test_model3 = pd.get_dummies(
    test_model3_data,
    drop_first=False
)

X_test_model3 = X_test_model3.reindex(
    columns=X_model3.columns,
    fill_value=0
)

print("Model 3 test shape:", X_test_model3.shape)

Model 3 test shape: (418, 11)


In [ ]:
model3_predictions = final_model3.predict(X_test_model3)

print("Tahmin sayısı:", len(model3_predictions))
print("İlk 10 tahmin:", model3_predictions[:10])

Tahmin sayısı: 418
İlk 10 tahmin: [0 1 0 0 1 0 1 0 1 0]


In [ ]:
model3_submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": model3_predictions
})

model3_submission_path = os.path.join(
    base_path,
    "titanic_my_model_submission_v2_model3.csv"
)

model3_submission.to_csv(
    model3_submission_path,
    index=False
)

print("Submission v2 kaydedildi:")
print(model3_submission_path)

Submission v2 kaydedildi:
/content/drive/MyDrive/Yapay Zeka Eğitimi 2026/KAGGLE_500_PUAN/01_Raw_Data/titanic_my_model_submission_v2_model3.csv


In [ ]:
print("Satır sayısı:", len(model3_submission))
print("Sütunlar:", model3_submission.columns.tolist())
print("Tahmin sınıfları:", sorted(model3_submission["Survived"].unique()))

Satır sayısı: 418
Sütunlar: ['PassengerId', 'Survived']
Tahmin sınıfları: [np.int64(0), np.int64(1)]


# Model 3 — Kaggle Submission Sonucu

## Sonuç

Model 3 validation veri setinde **%82,68 accuracy** elde etmiştir.

Ancak Kaggle submission sonucunda **0.77511** skor elde edilmiştir.

Önceki Model 2'nin Kaggle skoru **0.77751** olduğundan Model 3,
Kaggle gizli testinde Model 2'nin **0.00240** gerisinde kalmıştır.

### Değerlendirme

Model 3 validation performansında Model 2'den daha iyi olmasına
rağmen Kaggle testinde daha düşük sonuç vermiştir.

Bu sonuç, validation performansı ile gerçek/gizli test performansının
aynı olmayabileceğini göstermektedir.

Bu nedenle proje kapsamında en iyi Kaggle sonucu Model 2'ye ait
olan **0.77751** olarak belirlenmiştir.

### Deneysel Sonuç

Model 3, daha kontrollü feature engineering yaklaşımının
validation performansını artırabildiğini; ancak bunun Kaggle gizli
testinde otomatik olarak daha iyi sonuç anlamına gelmediğini
göstermiştir.